# Model Context Protocol (MCP)

**Khóa: Claude with the Anthropic API (Anthropic Skilljar)**

### Vấn đề
Ở bài Tool Use, mỗi công cụ bạn phải **tự viết tích hợp riêng**. Có 100 ứng dụng × 50 công cụ = phải viết 5000 tích hợp. Mỗi nhà phát triển lại viết lại từ đầu cho cùng một thứ (GitHub, Google Drive, Slack...).

### Giải pháp: MCP — "Cổng USB-C cho AI" 🔌
**Model Context Protocol** là một **chuẩn mở** (open standard) do Anthropic tạo ra để kết nối AI với công cụ & dữ liệu. Viết **một lần**, dùng **mọi nơi**.

> 🔑 Ý tưởng: thay vì mỗi app tự viết tích hợp riêng, ai cũng tuân theo *cùng một giao thức* → một "MCP server" cho GitHub dùng được cho Claude Desktop, Claude Code, app của bạn...


## 1. Kiến trúc MCP (3 thành phần)

```
┌─────────────┐     ┌─────────────┐     ┌──────────────┐
│  MCP Host   │────▶│ MCP Client  │────▶│  MCP Server  │
│ (Claude app)│     │ (cầu nối)   │     │ (GitHub, DB) │
└─────────────┘     └─────────────┘     └──────────────┘
```

- **MCP Host**: ứng dụng AI (Claude Desktop, Claude Code, app của bạn).
- **MCP Client**: nằm trong host, nói chuyện với server theo giao thức MCP.
- **MCP Server**: chương trình bọc một dịch vụ (GitHub, Slack, database...) và phơi bày ra 3 thứ:
  - **Tools** (công cụ — hành động Claude gọi được)
  - **Resources** (dữ liệu — tệp, bản ghi để đọc)
  - **Prompts** (mẫu prompt dựng sẵn)


## 2. Dùng MCP qua API (mcp_servers + mcp_toolset)

API cho phép Claude kết nối trực tiếp tới **MCP server từ xa**. Cần **2 tham số đi cùng nhau** + beta flag `mcp-client-2025-11-20`:

1. `mcp_servers`: khai báo server (`type`, `url`, `name`).
2. `tools`: một mục `mcp_toolset` trỏ tới server đó bằng `mcp_server_name`.

> ⚠️ Thiếu `mcp_toolset` sẽ bị lỗi validation. Mỗi server phải được tham chiếu bởi đúng một toolset.


In [ ]:
import anthropic
client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

# Ví dụ MINH HỌA cấu trúc (thay URL bằng MCP server thật của bạn)
response = client.beta.messages.create(
    model=MODEL,
    max_tokens=1024,
    betas=["mcp-client-2025-11-20"],
    mcp_servers=[
        {
            "type": "url",
            "url": "https://your-mcp-server.example.com/mcp",
            "name": "my-tools",
            # "authorization_token": "...",   # nếu server cần xác thực
        }
    ],
    tools=[
        {"type": "mcp_toolset", "mcp_server_name": "my-tools"}
    ],
    messages=[{"role": "user", "content": "Hãy dùng các công cụ có sẵn để giúp tôi."}],
)
print(response.content)

## 3. Xây một MCP Server đơn giản (bằng Python)

Dùng SDK `mcp`. Server dưới đây phơi bày 1 công cụ `add`. Đây là file **chạy riêng** (không phải trong notebook), Claude Desktop / Claude Code có thể kết nối tới.

```bash
pip install mcp
```


In [ ]:
# Lưu thành file vd_mcp_server.py rồi cấu hình vào Claude Desktop/Code
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Cộng hai số."""
    return a + b

@mcp.resource("greeting://{name}")
def greeting(name: str) -> str:
    """Trả về lời chào."""
    return f"Xin chào, {name}!"

if __name__ == "__main__":
    mcp.run()
'''
print(server_code)

## 4. MCP vs Tool Use thường — chọn cái nào?

| | Tool Use (tự viết) | MCP |
|---|---|---|
| Tích hợp | Viết riêng cho từng app | Viết 1 lần, dùng nhiều app |
| Tái sử dụng | Thấp | Cao (cộng đồng chia sẻ server) |
| Phù hợp | Công cụ riêng, đơn giản, nội bộ | Tích hợp chuẩn, chia sẻ, hệ sinh thái |

👉 **Tool Use** cho công cụ nhỏ gọn riêng của bạn. **MCP** khi muốn tích hợp chuẩn, tái sử dụng, hoặc kết nối tới hệ sinh thái server có sẵn (GitHub, Slack, Google Drive...).


---
## ✅ Tóm tắt

- **MCP** = chuẩn mở kết nối AI với công cụ/dữ liệu — "USB-C cho AI".
- 3 thành phần: **Host → Client → Server**.
- Server phơi bày: **Tools, Resources, Prompts**.
- Qua API: `mcp_servers` + `mcp_toolset` (beta `mcp-client-2025-11-20`), đi cùng nhau.
- Lợi ích lớn nhất: **viết một lần, dùng mọi nơi** + hệ sinh thái server chia sẻ.
